In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import TransformedTargetRegressor
import xgboost as xgb
import shap
import joblib
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="darkgrid")
print("All imports successful ✓")


In [ ]:
df = pd.read_csv('../data/spending.csv')

print("Shape:", df.shape)
print("\nColumn names:", df.columns.tolist())
print("\nData types:\n", df.dtypes)
print("\nNull values:\n", df.isnull().sum())
print("\nFirst 5 rows:")
df.head()

In [ ]:
# Fix column names — strip spaces
df.columns = df.columns.str.strip()

# Parse Date properly
df['Date'] = pd.to_datetime(df['Date'])

# Extract useful time features
df['Year']       = df['Date'].dt.year
df['Month']      = df['Date'].dt.month
df['DayOfMonth'] = df['Date'].dt.day
df['DayOfWeek']  = df['Date'].dt.dayofweek   # 0=Monday, 6=Sunday
df['WeekOfYear'] = df['Date'].dt.isocalendar().week.astype(int)

# Separate debits (actual spending) from credits (income, payments)
debits = df[df['Transaction Type'] == 'debit'].copy()
credits = df[df['Transaction Type'] == 'credit'].copy()

print(f"Total transactions: {len(df)}")
print(f"Debits (expenses): {len(debits)}")
print(f"Credits (income/payments): {len(credits)}")
print(f"\nDate range: {df['Date'].min().date()} → {df['Date'].max().date()}")
print(f"\nUnique categories:\n{debits['Category'].value_counts()}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Total spend per category
cat_spend = debits.groupby('Category')['Amount'].sum().sort_values(ascending=False)
cat_spend.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Total Spend by Category (All Time)', fontsize=13)
axes[0].set_xlabel('')
axes[0].set_ylabel('Total Amount ($)')
axes[0].tick_params(axis='x', rotation=45)

# Average transaction per category
cat_avg = debits.groupby('Category')['Amount'].mean().sort_values(ascending=False)
cat_avg.plot(kind='bar', ax=axes[1], color='coral', edgecolor='white')
axes[1].set_title('Average Transaction Size by Category', fontsize=13)
axes[1].set_xlabel('')
axes[1].set_ylabel('Avg Amount ($)')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../data/eda_category.png', dpi=150)
plt.show()
print("Top 5 categories by total spend:")
print(cat_spend.head())

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Monthly total spend
monthly = debits.groupby(['Year', 'Month'])['Amount'].sum().reset_index()
monthly['Period'] = monthly['Year'].astype(str) + '-' + monthly['Month'].astype(str).str.zfill(2)
axes[0].plot(monthly['Period'], monthly['Amount'], marker='o', color='steelblue', linewidth=2)
axes[0].set_title('Total Monthly Spending Over Time', fontsize=13)
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Total Spend ($)')
axes[0].tick_params(axis='x', rotation=45)
axes[0].fill_between(range(len(monthly)), monthly['Amount'], alpha=0.1, color='steelblue')

# Spending heatmap: month vs category
pivot = debits.groupby(['Month', 'Category'])['Amount'].sum().unstack(fill_value=0)
sns.heatmap(pivot, ax=axes[1], cmap='YlOrRd', fmt='.0f', annot=True, linewidths=0.5)
axes[1].set_title('Monthly Spend Heatmap by Category', fontsize=13)
axes[1].set_xlabel('Category')
axes[1].set_ylabel('Month')

plt.tight_layout()
plt.savefig('../data/eda_monthly.png', dpi=150)
plt.show()

In [ ]:
# Our ML problem:
# Given: year, month, category -> Predict: total spend for that month+category
# This is a regression problem on aggregated monthly data.
# Credit Card Payment is a payoff/transfer category, not day-to-day spend, so exclude it.

model_debits = debits[debits['Category'] != 'Credit Card Payment'].copy()

ml_df = model_debits.groupby(['Year', 'Month', 'Category'])['Amount'].sum().reset_index()
ml_df.columns = ['Year', 'Month', 'Category', 'MonthlySpend']

# Encode Category as a number (ML models need numbers, not strings)
le = LabelEncoder()
ml_df['CategoryEncoded'] = le.fit_transform(ml_df['Category'])

# Save the encoder - we'll need it in the API later
joblib.dump(le, '../data/label_encoder.pkl')

# Feature engineering: lag features (last month's spend for same category)
ml_df = ml_df.sort_values(['Category', 'Year', 'Month'])
ml_df['PrevMonthSpend'] = ml_df.groupby('Category')['MonthlySpend'].shift(1)
ml_df['PrevPrevMonthSpend'] = ml_df.groupby('Category')['MonthlySpend'].shift(2)
ml_df['RollingAvg3'] = ml_df.groupby('Category')['MonthlySpend'].transform(
    lambda x: x.shift(1).rolling(3, min_periods=1).mean()
)

# Drop rows where lag features are NaN (first month per category)
ml_df = ml_df.dropna()

print("ML dataset shape:", ml_df.shape)
print("Excluded from ML target: Credit Card Payment")
print()
print("Sample:")
ml_df.head(10)


In [ ]:
# Sort by TIME first (not category) before splitting
# This ensures all categories appear in both train and test
ml_df_time_sorted = ml_df.sort_values(['Year', 'Month', 'Category']).reset_index(drop=True)

split_idx = int(len(ml_df_time_sorted) * 0.8)
train_df = ml_df_time_sorted.iloc[:split_idx].copy()
test_df = ml_df_time_sorted.iloc[split_idx:].copy()

# Category statistics are computed from training data only to avoid leakage.
category_stats_df = train_df.groupby('Category')['MonthlySpend'].agg(
    CategoryMean='mean',
    CategoryMedian='median',
    CategoryStd='std'
).fillna(0)

global_category_stats = {
    'CategoryMean': float(train_df['MonthlySpend'].mean()),
    'CategoryMedian': float(train_df['MonthlySpend'].median()),
    'CategoryStd': float(train_df['MonthlySpend'].std())
}

for frame in (train_df, test_df):
    frame['CategoryMean'] = frame['Category'].map(category_stats_df['CategoryMean']).fillna(global_category_stats['CategoryMean'])
    frame['CategoryMedian'] = frame['Category'].map(category_stats_df['CategoryMedian']).fillna(global_category_stats['CategoryMedian'])
    frame['CategoryStd'] = frame['Category'].map(category_stats_df['CategoryStd']).fillna(global_category_stats['CategoryStd'])
    frame['LagDelta'] = frame['PrevMonthSpend'] - frame['PrevPrevMonthSpend']
    frame['LagToMeanRatio'] = frame['PrevMonthSpend'] / (frame['CategoryMean'] + 1)
    frame['MonthSin'] = np.sin(2 * np.pi * frame['Month'] / 12)
    frame['MonthCos'] = np.cos(2 * np.pi * frame['Month'] / 12)

features = [
    'Year', 'Month', 'CategoryEncoded',
    'PrevMonthSpend', 'PrevPrevMonthSpend', 'RollingAvg3',
    'CategoryMean', 'CategoryMedian', 'CategoryStd',
    'LagDelta', 'LagToMeanRatio', 'MonthSin', 'MonthCos'
]
target = 'MonthlySpend'

X_train, X_test = train_df[features], test_df[features]
y_train, y_test = train_df[target], test_df[target]

category_stats = {
    'by_category': category_stats_df.to_dict(orient='index'),
    'global': global_category_stats
}

print(f"Train size: {len(X_train)} | Test size: {len(X_test)}")
print(f"Categories in train: {train_df['Category'].nunique()} unique")
print(f"Categories in test: {test_df['Category'].nunique()} unique")


In [ ]:
def evaluate_model(name, model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)
    print(f"{name:20s} | MAE: {mae:8.2f} | RMSE: {rmse:8.2f} | R²: {r2:.4f}")
    return {'model': model, 'name': name, 'mae': mae, 'rmse': rmse, 'r2': r2, 'preds': preds}

# --- Global model on all spend categories ---
print("=== GLOBAL MODEL (spend categories) ===")
candidate_models = {
    'LinearRegression': Pipeline([
        ('scaler', StandardScaler()), ('model', LinearRegression())
    ]),
    'Ridge': Pipeline([
        ('scaler', StandardScaler()), ('model', Ridge(alpha=1.0))
    ]),
    'Lasso': Pipeline([
        ('scaler', StandardScaler()), ('model', Lasso(alpha=0.1))
    ]),
    'RandomForest': RandomForestRegressor(
        n_estimators=200, max_depth=4, min_samples_leaf=3,
        random_state=42, n_jobs=1
    ),
    'RandomForestLog': TransformedTargetRegressor(
        regressor=RandomForestRegressor(
            n_estimators=100, max_depth=6,
            random_state=42, n_jobs=1
        ),
        func=np.log1p,
        inverse_func=np.expm1
    ),
    'XGBoost': xgb.XGBRegressor(
        n_estimators=300, learning_rate=0.03, max_depth=3,
        subsample=0.8, colsample_bytree=0.8,
        min_child_weight=3, reg_alpha=0.1, reg_lambda=1.0,
        random_state=42, verbosity=0, n_jobs=1
    ),
}

results = {
    name: evaluate_model(name, model, X_train, y_train, X_test, y_test)
    for name, model in candidate_models.items()
}

# --- Per-category analysis on the best R² model ---
best_r2_name = max(results, key=lambda name: results[name]['r2'])
print()
print(f"=== PER-CATEGORY MAE ({best_r2_name}) ===")
best_r2_model = results[best_r2_name]['model']
per_category_df = test_df.copy()
per_category_df['Predicted'] = best_r2_model.predict(X_test)
per_category_df['Error'] = abs(per_category_df['MonthlySpend'] - per_category_df['Predicted'])

per_cat = per_category_df.groupby('Category').agg(
    Actual_Mean=('MonthlySpend', 'mean'),
    Predicted_Mean=('Predicted', 'mean'),
    MAE=('Error', 'mean')
).round(2).sort_values('MAE')

print(per_cat.to_string())
print()
print(f"Best predicted category: {per_cat.index[0]}")
print(f"Worst predicted category: {per_cat.index[-1]}")


In [ ]:
# Build a clean comparison table
comparison = pd.DataFrame([
    {
        'Model': name,
        'MAE ($)': round(r['mae'], 2),
        'RMSE ($)': round(r['rmse'], 2),
        'R²': round(r['r2'], 4)
    }
    for name, r in results.items()
]).sort_values('R²', ascending=False).reset_index(drop=True)

print("=== MODEL COMPARISON ===")
print(comparison.to_string(index=False))

# Plot comparison
fig, ax = plt.subplots(figsize=(10, 5))
colors = plt.cm.Set2(np.linspace(0, 1, len(comparison)))
bars = ax.bar(comparison['Model'], comparison['R²'], color=colors)
ax.set_title('Model Comparison - R² (higher is better)', fontsize=13)
ax.set_ylabel('R² Score')
ax.tick_params(axis='x', rotation=15)
for bar, val in zip(bars, comparison['R²']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01, f'{val}',
            ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.savefig('../data/model_comparison.png', dpi=150)
plt.show()

# Pick best model by highest R²
best_name = comparison.iloc[0]['Model']
best_model = results[best_name]['model']
print(f"Best model: {best_name} with R² = {comparison.iloc[0]['R²']} and MAE = ${comparison.iloc[0]['MAE ($)']}")


In [ ]:
# SHAP tells you: "for this prediction, which features pushed it up or down?"
# Use the plain Random Forest candidate for a tree-based explanation plot.
explainer = shap.TreeExplainer(results['RandomForest']['model'])
shap_values = explainer.shap_values(X_test)

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test, feature_names=features, show=False)
plt.title('SHAP Feature Importance - Random Forest')
plt.tight_layout()
plt.savefig('../data/shap_importance.png', dpi=150, bbox_inches='tight')
plt.show()

# Save the selected model and metadata
joblib.dump(best_model, '../data/model.pkl')
joblib.dump(best_name, '../data/best_model_name.pkl')
joblib.dump(category_stats, '../data/category_stats.pkl')

# Save feature list (API needs to know the exact feature order)
joblib.dump(features, '../data/features.pkl')

# Save comparison table for the API to serve to frontend
comparison.to_csv('../data/model_comparison.csv', index=False)

print("Model saved to ../data/model.pkl")
print("Best model name saved to ../data/best_model_name.pkl")
print("Category stats saved to ../data/category_stats.pkl")
print("Label encoder saved to ../data/label_encoder.pkl")
print("Feature list saved to ../data/features.pkl")
print("Comparison table saved to ../data/model_comparison.csv")
print("Training complete.")
